# Colombia Open — Demografía

Este notebook muestra cómo cargar el dataset **`demografia`** (proyecciones de población municipal 2018-2042, DANE) con `colombia-open`: serie por municipio, año y área geográfica, y cómo unirlo con `municipios`.

> **Nota (2026-10):** `colombia-open` ya está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente (ver `README.md` del repo, sección "Estado del proyecto").

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
demografia = hub.load_polars("demografia")

print(demografia.shape)
demografia.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `cod_dpto` | string (2 dígitos) | Código DIVIPOLA de departamento. |
| `nombre_dpto` | string | Nombre del departamento. |
| `cod_mpio` | string (5 dígitos) | Join key hacia `municipios`. **Nunca se convierte a int.** |
| `nombre_mpio` | string | Nombre del municipio. Las áreas no municipalizadas llevan el sufijo `(ANM)`. |
| `anio` | int | Año de la proyección, 2018 a 2042. |
| `area_geografica` | string | `"Cabecera Municipal"`, `"Centros Poblados y Rural Disperso"` o `"Total"` (la suma de las otras dos, publicada tal cual por el DANE). |
| `poblacion` | int | Población proyectada para esa combinación municipio/año/área. |

La clave primaria es compuesta: `(cod_mpio, anio, area_geografica)`.

## Serie de tiempo de un municipio

In [ ]:
# Población total de Medellín (cod_mpio == "05001"), 2018-2042
demografia.filter(
    (demografia["cod_mpio"] == "05001") & (demografia["area_geografica"] == "Total")
).select(["anio", "poblacion"]).sort("anio")

## Join con `municipios`

`cross_view()` cruza por `cod_mpio` por defecto — funciona directo entre `demografia` y `municipios` sin pasar `on=`.

In [ ]:
joined = hub.cross_view(["demografia", "municipios"])
joined.filter(
    (joined["cod_mpio"] == "76001") & (joined["anio"] == 2030) & (joined["area_geografica"] == "Total")
)

## Validación e integridad referencial

`validate_dataset()` resuelve automáticamente la referencia declarada en el registro (`cod_mpio` -> `municipios`). `demografia` y `municipios` son dos productos del DANE con ciclos de actualización distintos, así que un código histórico/absorbido (`94663`, Mapiripana) se reporta como **warning**, no como error — ver `docs/datasets/demografia.md`.

In [ ]:
result = hub.validate_dataset("demografia")
print("status:", result["status"])
for warning in result["warnings"]:
    print("warning:", warning)

## Limitaciones conocidas

- Un código de municipio (`94663`) no existe en el `municipios` vigente — absorbido por Barrancominas desde 2019, conservado en la serie histórica.
- `area_geografica == "Total"` es redundante por diseño: se publica tal cual la entrega el DANE, no se deriva en el pipeline.
- Sin desagregación por sexo o grupo de edad.

Documentación completa del esquema: [`docs/datasets/demografia.md`](../../docs/datasets/demografia.md).